##### lerobot の ping pong の移し替えの model (aloha_static_pingpong) を簡略した、学習用の pytorch コードです。  
学習データは、実際のデータを huggingface からダウンロードして使います。  
複雑な、lerobot の train が、実際にしていることを、エミューレートして、pytorch の model と同じだよと  
言うことの材料になれば、幸いです。  
  
本コードを作成するにあたっては、Google Ai に、大変手助けをもらいました。  
ありがとうと、感謝します。  


In [1]:
import os
import sys
from pathlib import Path

#export PYTHONPATH=$PYTHONPATH:/home/nishi/local/git-download/lerobot/src
# パスを通す（お使いの環境に合わせて絶対パスに書き換えてください）
# git clone をした、パスを使ってください。
lerobot_path = "/home/nishi/local/git-download/lerobot"
sys.path.append(os.path.join(lerobot_path, "src"))

from lerobot.configs.train import TrainPipelineConfig
from lerobot.scripts.lerobot_train import train
from lerobot.configs.policies import PreTrainedConfig

batch_size=4

### オリジナルの ping pong 移し替えの model を、学習用にシンプルにした、model です。  
inputs:  
camera 入力が、1 個になります。  
オリジナルは、2個みたい。  
アームのセンサーは、同じ 14 個 だと思う。  
outputs:  
アームの 14 個のサーボの位置を示す数値データ。  

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights

# オリジナルの ping pong 移し替えの model を、学習用にシンプルにした、model です。
class SimpleACT(nn.Module):
    def __init__(self, num_joints=14, num_queries=100):
        super().__init__()
        
        # 1. 視覚神経：ResNet18（学習済み重みを利用）
        # weights=... とすることで、最初から「形」を捉える能力を持たせます
        weights = ResNet18_Weights.IMAGENET1K_V1
        self.backbone = resnet18(weights=weights)
        self.backbone.fc = nn.Identity() # 最後の1000クラス分類層を削除
        
        # 2. 現在地を知る神経
        self.state_proj = nn.Linear(num_joints, 512)
        
        # 3. 思考回路 (Transformer)
        # 構造をシンプルに（層を少なく）して、GTX 1070での学習速度を優先します
        self.transformer = nn.Transformer(
            d_model=512, 
            nhead=8, 
            num_encoder_layers=2, # 教材用に4層から2層へ軽量化
            num_decoder_layers=2,
            batch_first=True      # データ形式を扱いやすくします
        )
        
        # 4. 未来を予測するための「100個の質問票」
        self.query_embed = nn.Embedding(num_queries, 512)
        
        # 5. 命令の出口 (14個の角度へ)
        self.action_head = nn.Linear(512, num_joints)

    def forward(self, image, current_state):
        # A. 画像サイズを 224x224 にリサイズ（ResNetの標準サイズ）
        # 元の 480x640 から変換することで計算を速めます
        image = F.interpolate(image, size=(224, 224), mode='bilinear', align_corners=False)
        
        # B. 特徴抽出 [Batch, 512]
        img_feat = self.backbone(image)
        
        # C. 角度データの変換 [Batch, 512]
        state_feat = self.state_proj(current_state)
        
        # D. 視覚と現在地を結合 [Batch, 2, 512]
        # Transformerへの「今の状況説明」
        combined_feat = torch.stack([img_feat, state_feat], dim=1)
        
        # E. 未来の枠組みを用意 [Batch, 100, 512]
        query = self.query_embed.weight.unsqueeze(0).repeat(image.size(0), 1, 1)
        
        # F. Transformer思考 [Batch, 100, 512]
        hs = self.transformer(combined_feat, query)
        
        # G. 最終命令 [Batch, 100, 14]
        actions = self.action_head(hs)
        
        return actions

### dataset に delta_timestamps を挿入します。  
「ACTモデルの真価を発揮するため、現在から未来100ステップ分（約2秒間）の動作を一気に学習するように設定しました。
これにより、一貫性のある滑らかな動きが可能になります。」


In [3]:
import matplotlib.pyplot as plt
import japanize_matplotlib # これを足すだけで日本語が使えるようになります
import torch

from lerobot.datasets.lerobot_dataset import LeRobotDataset
# データセットを直接読み込む
#dataset = LeRobotDataset("lerobot/aloha_static_pingpong_test")

# データセットを作るコードを探して、delta_timestamps を追加します
dataset = LeRobotDataset(
    "lerobot/aloha_static_pingpong_test",
    # 現在(0)から未来へ、100ステップ分（適当な間隔で）取得する設定
    delta_timestamps={
        "action": [i/50 for i in range(100)], # 50Hzの場合、2秒分
    }
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
from torch.utils.data import DataLoader

# 1. 教科書（DataLoader）の準備
# num_workers=0 は先ほどのエラー回避のためです
train_loader = DataLoader(
    dataset, 
    batch_size=batch_size,
    shuffle=True, 
    num_workers=0
)

from torchvision import transforms
# 1. 特訓メニュー（オーグメンテーション）の定義
train_transform = transforms.Compose([
    # 明るさ、コントラスト、彩度、色相をランダムに変える（照明変化への耐性）
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    # ランダムに少しだけぼかす（レンズの汚れやピントズレへの耐性）
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
    ], p=0.3), 
])


In [5]:
#2. 特訓ループ：1コマの「完コピ」実験
import torch.optim as optim
import torch
import numpy as np

out_dir="output"
# 出力先のクラスフォルダを作成
if not os.path.exists(out_dir):
    os.makedirs(out_dir)
best_path = os.path.join(out_dir, "pingpong_symple_best.pth")

# 1. 脳（SimpleACT）の準備
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SimpleACT().to(device)
model.train()

# 2. 工具（Optimizer）の設定
# 1コマ集中特訓なので、少し強めの学習率(1e-4)で一気に覚えさせます
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

if False:
    # 3. 特訓データの準備 (1コマ分)
    frame = dataset[0]
    img = frame["observation.images.cam_high"].unsqueeze(0).to(device)   # [1, 3, 480, 640]
    state = frame["observation.state"].unsqueeze(0).to(device)          # [1, 14]
    # お手本（Action）を [1, 100, 14] の形に整える
    target = frame["action"].unsqueeze(0).unsqueeze(1).repeat(1, 100, 1).to(device)

else:
    # 2. 特訓ループ（とりあえず100回回してみる）
    # iter(train_loader) で教科書からランダムにページをめくります
    loader_iter = iter(train_loader)

print(f"特訓開始 (Device: {device})...")

losses = []
#num_epochs=201
num_epochs=1

best_loss = float('inf')
min_loss=0.1
counter=0

for epoch in range(num_epochs):
    total_loss = 0
    losses = []
    for step,batch_data in enumerate(train_loader):
        # データを整理してGPUへ
        #img = batch_data["observation.images.cam_high"].to(device)
        # batch_data から画像を取り出した直後に適用
        img_raw = batch_data["observation.images.cam_high"].to(device)
        img_augmented = train_transform(img_raw)
        
        state = batch_data["observation.state"].to(device)
        # 未来100ステップの正解
        target = batch_data["action"].to(device)
        #result = x.unsqueeze(1).repeat(1, 100, 1)
        #target = batch_data["action"].unsqueeze(1).repeat(1, 100, 1).to(device)
        #print('target.shape:',target.shape)
        # batch_data の中身
        #target = batch_data["action"]        # [Batch, 100, 14]（最後は同じ値が並ぶ）
        is_pad = batch_data["action_is_pad"].to(device) # [Batch, 100]（本物ならFalse, 埋めた場所はTrue）

        
        # --- 学習の4ステップ ---
        # A. 予測
        #pred = model(img, state)
        pred = model(img_augmented, state)
        #print('pred.shape:',pred.shape)
        
        # B. ズレ(Loss)の計算
        #loss = F.l1_loss(pred, target)
        # プロレベルのLoss計算のイメージ
        loss = F.l1_loss(pred, target, reduction="none") # 一旦バラバラに計算
        loss = loss[~is_pad].mean() # 本物のデータがある場所だけの平均を取る
        
        # C. 反省
        optimizer.zero_grad()
        loss.backward()
        
        # D. 改善（脳の更新）
        optimizer.step()
        
        # 進捗表示
        losses.append(loss.item())
        #total_loss += loss.item()
        if step % 200 == 0:
            print(f"Step {step:3d} | Loss: {loss.item():.6f}")

    #avg_loss = total_loss / len(train_loader)
    avg_loss = np.sum(losses) / len(losses)
    print(f"epoch {epoch:3d} | Loss: {avg_loss.item():.6f}")

    # 一番良いモデルを保存
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), best_path)
        print(f"Model saved! (Best Loss: {best_loss:.4f})")
        counter=0

print("\n--- 特訓完了！ ---")

if False:
    num_steps=201
    for step in range(num_steps):
        if True:
            # 教科書から「今のバッチ」を1組取り出す
            batch_data = next(loader_iter)
            
            # データを整理してGPUへ
            img = batch_data["observation.images.cam_high"].to(device)
            state = batch_data["observation.state"].to(device)
            # 未来100ステップの正解
            #target = batch_data["action"].to(device)
            #result = x.unsqueeze(1).repeat(1, 100, 1)
            target = batch_data["action"].unsqueeze(1).repeat(1, 100, 1).to(device)
            
        # --- 学習の4ステップ ---
        # A. 予測
        pred = model(img, state)
        #print('pred.shape:',pred.shape)
        
        # B. ズレ(Loss)の計算
        loss = F.l1_loss(pred, target)
        # プロレベルのLoss計算のイメージ
        #loss = F.l1_loss(pred, target, reduction="none") # 一旦バラバラに計算
        #loss = loss[~is_pad].mean() # 本物のデータがある場所だけの平均を取る
        
        # C. 反省
        optimizer.zero_grad()
        loss.backward()
        
        # D. 改善（脳の更新）
        optimizer.step()
        
        # 進捗表示
        losses.append(loss.item())
        #if step % 100 == 0:
        if step % 20 == 0:
            print(f"Step {step:3d} | Loss: {loss.item():.6f}")
    
    print("\n--- 特訓完了！ ---")
    
    # 4. 成果のグラフ表示
    import matplotlib.pyplot as plt
    plt.plot(losses)
    plt.title("学習曲線 (Lossの低下)")
    plt.xlabel("Step")
    plt.ylabel("Loss (ズレ)")
    plt.show()

特訓開始 (Device: cuda)...
Step   0 | Loss: 0.676257
Step 200 | Loss: 0.164917
Step 400 | Loss: 0.142336
Step 600 | Loss: 0.160130
Step 800 | Loss: 0.086644
Step 1000 | Loss: 0.077532
Step 1200 | Loss: 0.075792
Step 1400 | Loss: 0.069972
epoch   0 | Loss: 0.118188
Model saved! (Best Loss: 0.1182)

--- 特訓完了！ ---


In [6]:
latest_path = os.path.join(out_dir, "latest_model.pth")
# --- 5. 最後にモデルを保存 (Final Model) ---
torch.save(model.state_dict(), latest_path)
print(f"最終モデルを保存しました: {latest_path}")


最終モデルを保存しました: output/latest_model.pth


In [ ]:
# 学習に使っていない可能性が高い「最後の方のエピソード」から1コマ取る
test_frame = dataset[len(dataset) - 1] 

img_t = test_frame["observation.images.cam_high"].unsqueeze(0).to(device)
state_t = test_frame["observation.state"].unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    prediction = model(img_t, state_t)

print("未知の画像に対するAIの予測:\n", prediction[0, 0, :7])
print("\n実際のお手本データ:\n", test_frame["action"][:7])
